In [2]:
import requests
import time
from src.config import OPENALEX_API_KEY

TU_WIEN    = "I57206974"
ETH_ZURICH = "I114027177"
TU_BERLIN  = "I63966007"


def _get(url, params=None):
    if params is None:
        params = {}
    if OPENALEX_API_KEY:
        params["api_key"] = OPENALEX_API_KEY
    time.sleep(0.12)
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    return r.json()


In [3]:
# ── part 1: confirm institution ids ──────────────────────────────────────────

def check_institutions():
    for name in ["TU Wien", "ETH Zurich", "TU Berlin"]:
        data = _get("https://api.openalex.org/institutions", {
            "search": name, "per_page": 2,
            "select": "id,display_name,country_code,works_count",
        })
        print(f"\n{name}")
        for r in data["results"]:
            print(f"  {r['id']}  |  {r['display_name']}  |  {r['country_code']}  |  {r['works_count']} works")

In [4]:
# ── part 2: look at a few papers and see what fields come back ────────────────

def look_at_works(inst_id=TU_WIEN, year=2022, n=3):
    data = _get("https://api.openalex.org/works", {
        "filter": f"authorships.institutions.id:{inst_id},publication_year:{year},type:article",
        "per_page": n,
        "sort": "cited_by_count:desc",
    })
    print(f"\ntotal matching: {data['meta']['count']}")
    for work in data["results"]:
        print(f"\n  title:    {work['title']}")
        print(f"  id:       {work['id']}")
        print(f"  year:     {work['publication_year']}")
        print(f"  cited by: {work['cited_by_count']}")
        for a in work.get("authorships", [])[:2]:
            name = a.get("author", {}).get("display_name", "?")
            insts = [i.get("display_name", "?") for i in a.get("institutions", [])]
            print(f"    author: {name}  @  {insts}")
        for c in work.get("concepts", [])[:3]:
            print(f"    concept: {c['display_name']}  ({c['score']:.2f})")
        refs = work.get("referenced_works", [])
        print(f"  references: {len(refs)}  — first few: {refs[:3]}")

In [5]:
# ── part 3: quick citation overlap between two institutions ───────────────────

def check_citation_overlap(inst_a=TU_WIEN, inst_b=ETH_ZURICH, year=2022, n=50):
    def fetch(inst_id):
        data = _get("https://api.openalex.org/works", {
            "filter": f"authorships.institutions.id:{inst_id},publication_year:{year},type:article",
            "per_page": n, "sort": "cited_by_count:desc",
            "select": "id,referenced_works",
        })
        return data["results"]

    works_a = fetch(inst_a)
    works_b = fetch(inst_b)
    ids_a = {w["id"] for w in works_a}
    ids_b = {w["id"] for w in works_b}

    a_cites_a = sum(1 for w in works_a for ref in w.get("referenced_works", []) if ref in ids_a)
    a_cites_b = sum(1 for w in works_a for ref in w.get("referenced_works", []) if ref in ids_b)
    b_cites_a = sum(1 for w in works_b for ref in w.get("referenced_works", []) if ref in ids_a)

    total = a_cites_a + a_cites_b
    score = round(a_cites_a / total, 3) if total > 0 else 0

    print(f"\ncitation overlap ({year}, top {n} papers each)")
    print(f"  a→a internal: {a_cites_a}")
    print(f"  a→b external: {a_cites_b}")
    print(f"  b→a external: {b_cites_a}")
    print(f"  rough bubble score for a: {score}")




In [6]:
# ── part 4: find concept id for a research field ─────────────────────────────

def find_concept(query="knowledge graph"):
    data = _get("https://api.openalex.org/concepts", {
        "search": query, "per_page": 5,
        "select": "id,display_name,level,works_count",
    })
    print(f"\n'{query}'")
    for c in data["results"]:
        print(f"  {c['id']}  |  {c['display_name']}  |  level {c['level']}  |  {c['works_count']:,} works")

In [7]:
# ── run ───────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    check_institutions()
    look_at_works(inst_id=TU_WIEN, year=2022, n=3)
    check_citation_overlap(inst_a=TU_WIEN, inst_b=ETH_ZURICH, year=2022, n=50)
    find_concept("knowledge graph")
    find_concept("machine learning")


TU Wien
  https://openalex.org/I145847075  |  TU Wien  |  AT  |  105341 works

ETH Zurich
  https://openalex.org/I35440088  |  ETH Zurich  |  CH  |  236966 works

TU Berlin
  https://openalex.org/I4577782  |  Technische Universität Berlin  |  DE  |  109659 works

total matching: 7997

  title:    New insights into the genetic etiology of Alzheimer’s disease and related dementias
  id:       https://openalex.org/W4223591189
  year:     2022
  cited by: 2402
    author: Céline Bellenguez  @  ['Inserm', 'Université de Lille', 'Institut Pasteur de Lille', 'Centre Hospitalier Universitaire de Lille', 'Facteurs de risque et déterminants moléculaires des maladies liées au vieillissement']
    author: Fahri Küçükali  @  ['University of Antwerp', 'VIB-UAntwerp Center for Molecular Neurology']
    concept: Biology  (0.82)
    concept: Etiology  (0.69)
    concept: Disease  (0.66)
  references: 104  — first few: ['https://openalex.org/W1964895626', 'https://openalex.org/W1965552543', 'https://op